In [8]:


import subprocess
import sys
import os
from pathlib import Path



import pandas as pd
import numpy as np
from datasets import load_dataset, DatasetDict

# ------------------------------------------------------------
# 1. Detect and load the dataset
# ------------------------------------------------------------
def load_parallel_text_files(eng_path, hi_path):
    """
    Reads two text files line by line and returns a DataFrame with columns
    'english' and 'hindi'. Skips lines that are empty after stripping.
    Both files must have the same number of lines.
    """
    with open(eng_path, "r", encoding="utf-8") as f_en, open(hi_path, "r", encoding="utf-8") as f_hi:
        en_lines = [line.strip() for line in f_en]
        hi_lines = [line.strip() for line in f_hi]

    if len(en_lines) != len(hi_lines):
        print(f"WARNING: File lengths differ: eng={len(en_lines)}, hin={len(hi_lines)}. "
              f"Truncating to minimum length.")
        min_len = min(len(en_lines), len(hi_lines))
        en_lines = en_lines[:min_len]
        hi_lines = hi_lines[:min_len]

    # Remove pairs where either side is empty
    pairs = [(en, hi) for en, hi in zip(en_lines, hi_lines) if en and hi]
    print(f"Loaded {len(pairs)} non‑empty sentence pairs from '{eng_path}' and '{hi_path}'.")
    return pd.DataFrame(pairs, columns=["english", "hindi"])

def detect_and_load(path):
    """
    Given a path (directory or file), detect the dataset format and return
    a pandas DataFrame containing all rows from all splits.
    """
    path = Path(path)
    if path.is_dir():
        # Hugging Face dataset directory (contains dataset_dict.json)
        if (path / "dataset_dict.json").exists():
            print("Detected Hugging Face Dataset directory (dataset_dict.json).")
            ds = load_dataset(str(path))
            return _hf_to_dataframe(ds)
        # Parquet files
        parquet_files = list(path.glob("*.parquet"))
        if parquet_files:
            print(f"Detected Parquet files: {len(parquet_files)} file(s).")
            dfs = [pd.read_parquet(f) for f in parquet_files]
            return pd.concat(dfs, ignore_index=True)
        # CSV files
        csv_files = list(path.glob("*.csv"))
        if csv_files:
            print(f"Detected CSV files: {len(csv_files)} file(s).")
            dfs = [pd.read_csv(f) for f in csv_files]
            return pd.concat(dfs, ignore_index=True)
        # JSON Lines
        jsonl_files = list(path.glob("*.jsonl"))
        if jsonl_files:
            print(f"Detected JSONL files: {len(jsonl_files)} file(s).")
            dfs = [pd.read_json(f, lines=True) for f in jsonl_files]
            return pd.concat(dfs, ignore_index=True)
        # JSON array/records
        json_files = list(path.glob("*.json"))
        if json_files:
            print(f"Detected JSON files: {len(json_files)} file(s).")
            dfs = [pd.read_json(f) for f in json_files]
            return pd.concat(dfs, ignore_index=True)
        # Arrow/Feather files
        arrow_files = list(path.glob("*.arrow"))
        if arrow_files:
            print(f"Detected Arrow files: {len(arrow_files)} file(s).")
            dfs = [pd.read_feather(f) for f in arrow_files]
            return pd.concat(dfs, ignore_index=True)

        raise FileNotFoundError(f"No supported dataset files found in {path}")

    elif path.is_file():
        suffix = path.suffix.lower()
        if suffix == ".parquet":
            return pd.read_parquet(path)
        elif suffix == ".csv":
            return pd.read_csv(path)
        elif suffix in (".jsonl", ".json"):
            return pd.read_json(path, lines=suffix == ".jsonl")
        elif suffix in (".arrow", ".feather"):
            return pd.read_feather(path)
        else:
            raise ValueError(f"Unsupported file format: {suffix}")
    else:
        raise FileNotFoundError(f"Path does not exist: {path}")

def _hf_to_dataframe(dataset):
    """Convert a Hugging Face Dataset/DatasetDict to a single DataFrame."""
    if isinstance(dataset, DatasetDict):
        frames = []
        for split_name, split_ds in dataset.items():
            df = split_ds.to_pandas()
            df["_split"] = split_name
            frames.append(df)
        return pd.concat(frames, ignore_index=True)
    else:
        return dataset.to_pandas()

# ------------------------------------------------------------
# 2. Identify English and Hindi columns (for the generic load case)
# ------------------------------------------------------------
def identify_lang_columns(df):
    """
    Return (english_col, hindi_col) based on keyword matching.
    """
    cols = df.columns.tolist()
    en_keywords = ["english", "en", "source", "src", "eng"]
    hi_keywords = ["hindi", "hi", "target", "tgt", "hin", "dest"]

    def find_col(keywords, exclude=None):
        for kw in keywords:
            for col in cols:
                if kw in col.lower():
                    if exclude is None or col != exclude:
                        return col
        return None

    en_col = find_col(en_keywords)
    hi_col = find_col(hi_keywords, exclude=en_col)

    # Fallback: exactly 2 columns → first English, second Hindi
    if en_col is None and hi_col is None and len(cols) == 2:
        en_col, hi_col = cols[0], cols[1]
        print(f"Guessing columns: '{en_col}' for English, '{hi_col}' for Hindi.")
    elif en_col is None:
        raise ValueError("Could not identify English column. Available: " + ", ".join(cols))
    elif hi_col is None:
        raise ValueError("Could not identify Hindi column. Available: " + ", ".join(cols))

    return en_col, hi_col

# ------------------------------------------------------------
# 3. Text cleaning
# ------------------------------------------------------------
def clean_text(text):
    """
    Strip, replace all whitespace (tabs, newlines) with single space,
    collapse multiple spaces. Return None if empty after stripping.
    """
    if not isinstance(text, str):
        return None
    text = text.strip()
    if not text:
        return None
    text = " ".join(text.split())
    return text

# ------------------------------------------------------------
# 4. Word & character counting
# ------------------------------------------------------------
def word_count(text):
    return len(text.split())

def char_count(text):
    return len(text)

# ------------------------------------------------------------
# 5. Main processing pipeline
# ------------------------------------------------------------
def main():
    # --- Priority 1: Look for eng.txt / hin.txt in the current directory ---
    eng_file = Path("/content/eng.txt")
    hin_file = Path("/content/hin.txt")
    if eng_file.exists() and hin_file.exists():
        print("Found 'eng.txt' and 'hin.txt' in the current directory. Loading parallel text files...")
        df = load_parallel_text_files(eng_file, hin_file)
    else:
        # Fallback: dataset path from command line or current directory
        if len(sys.argv) > 1:
            dataset_path = sys.argv[1]
        else:
            dataset_path = "."
        print(f"Loading dataset from: {dataset_path}")
        df = detect_and_load(dataset_path)

    print(f"Original dataset shape: {df.shape}")

    # --- Inspection (Step 2) ---
    if "_split" in df.columns:
        print("\nSplits found:")
        for split in df["_split"].unique():
            count = (df["_split"] == split).sum()
            print(f"  {split}: {count} rows")
    else:
        print("No split information; treating as single dataset.")

    print(f"\nColumn names: {list(df.columns)}")
    print(f"Total rows: {len(df)}")
    print("\nFirst 5 rows:")
    print(df.head(5).to_string())

    # Identify language columns if they are not already 'english'/'hindi'
    if "english" not in df.columns or "hindi" not in df.columns:
        en_col, hi_col = identify_lang_columns(df)
        print(f"\nUsing columns: English='{en_col}', Hindi='{hi_col}'")
        df = df[[en_col, hi_col]].copy()
        df.rename(columns={en_col: "english", hi_col: "hindi"}, inplace=True)
    else:
        # Already have the required column names from parallel text files
        en_col, hi_col = "english", "hindi"
        print(f"\nUsing columns: English='{en_col}', Hindi='{hi_col}'")
        df = df[["english", "hindi"]].copy()

    # Preprocessing (Step 3)
    df["english"] = df["english"].apply(clean_text)
    df["hindi"] = df["hindi"].apply(clean_text)

    # Remove rows where either is None
    before = len(df)
    df.dropna(subset=["english", "hindi"], inplace=True)
    after_clean = len(df)
    print(f"\nRows after removing empty/NaN sentences: {after_clean} "
          f"(dropped {before - after_clean} rows)")

    # Compute word counts
    df["wc_en"] = df["english"].apply(word_count)
    df["wc_hi"] = df["hindi"].apply(word_count)

    # Filter: 5 ≤ word count ≤ 70
    mask_wc = (df["wc_en"] >= 5) & (df["wc_en"] <= 70) & \
              (df["wc_hi"] >= 5) & (df["wc_hi"] <= 70)
    wc_filtered = df[mask_wc].copy()
    print(f"Rows after word‑count filter (5–70): {len(wc_filtered)} "
          f"(dropped {len(df) - len(wc_filtered)} rows)")

    # Word‑count difference
    wc_filtered["wc_diff"] = wc_filtered["wc_en"] - wc_filtered["wc_hi"]
    mask_diff = (wc_filtered["wc_diff"] >= -10) & (wc_filtered["wc_diff"] <= 10)
    diff_filtered = wc_filtered[mask_diff].copy()
    print(f"Rows after word‑count difference filter (−10…+10): {len(diff_filtered)} "
          f"(dropped {len(wc_filtered) - len(diff_filtered)} rows)")

    # Character counts
    diff_filtered["cc_en"] = diff_filtered["english"].apply(char_count)
    diff_filtered["cc_hi"] = diff_filtered["hindi"].apply(char_count)

    # Build final columns exactly as required
    final = diff_filtered.copy()
    # Column 5: Difference between Word Count (English) and Character Count (English)
    final["diff_wc_en_cc_en"] = final["wc_en"] - final["cc_en"]
    # Column 7: Difference between Character Count (English) and Character Count (Hindi)
    final["diff_cc_en_cc_hi"] = final["cc_en"] - final["cc_hi"]

    # Rename to required column names
    final.rename(columns={
        "wc_en": "Word Count (English)",
        "wc_hi": "Word Count (Hindi)",
        "cc_hi": "Character Count (Hindi)",
        "diff_wc_en_cc_en": "Difference between Word Count (English) and Character Count (English)",
        "diff_cc_en_cc_hi": "Difference between Character Count (English) and Character Count (Hindi)"
    }, inplace=True)

    # Select and order the seven required columns
    required_cols = [
        "english",
        "hindi",
        "Word Count (English)",
        "Word Count (Hindi)",
        "Difference between Word Count (English) and Character Count (English)",
        "Character Count (Hindi)",
        "Difference between Character Count (English) and Character Count (Hindi)"
    ]
    final = final[required_cols]

    print(f"\nFinal dataset size: {len(final)} rows")
    print("\nSample of final dataset (first 5 rows):")
    print(final.head(5).to_string())

    # --- Final validation ---
    assert final["english"].notna().all(), "Null English sentences remain"
    assert final["hindi"].notna().all(), "Null Hindi sentences remain"
    assert (final["Word Count (English)"].between(5, 70)).all(), "English word count out of range"
    assert (final["Word Count (Hindi)"].between(5, 70)).all(), "Hindi word count out of range"
    wc_diff = final["Word Count (English)"] - final["Word Count (Hindi)"]
    assert wc_diff.between(-10, 10).all(), "Word‑count difference out of range"
    print("\n✓ All final constraints satisfied.")

    # Export to Excel
    output_file = "cleaned_dataset.xlsx"
    final.to_excel(output_file, index=False, engine="openpyxl")
    print(f"\nExported cleaned dataset to '{output_file}'")

if __name__ == "__main__":
    main()

Found 'eng.txt' and 'hin.txt' in the current directory. Loading parallel text files...
Loaded 10000 non‑empty sentence pairs from '/content/eng.txt' and '/content/hin.txt'.
Original dataset shape: (10000, 2)
No split information; treating as single dataset.

Column names: ['english', 'hindi']
Total rows: 10000

First 5 rows:
                                                                                                                                        english                                                                                                                                                       hindi
0  However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles          आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।
1  Whosoever desires the reward of the world, with Allah is the reward of

In [10]:
pip install sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.2 MB/s eta 0:00:00


In [9]:


import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from sacrebleu import corpus_bleu, corpus_chrf, corpus_ter
import os


# Path to your Assignment 1 dataset (Excel or CSV)
DATASET_PATH = "/content/cleaned_dataset.xlsx"  # change to .csv if needed
# Number of sentences to use
N_SENTENCES = 100

# Model choice (free, open-source, not Helsinki-NLP or Google Translate)
# facebook/nllb-200-distilled-600M is a good multilingual model
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "eng_Latn"      # NLLB language code for English
TGT_LANG = "hin_Deva"      # NLLB language code for Hindi

# Output files
OUTPUT_EXCEL = "translations.xlsx"
OUTPUT_SCORES = "scores.txt"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================
# 1. Load and select 100 sentences
# ============================
# Adjust the reading method depending on file extension
if DATASET_PATH.endswith('.csv'):
    df = pd.read_csv(DATASET_PATH)
else:
    df = pd.read_excel(DATASET_PATH)

# Ensure required columns exist
assert 'english' in df.columns, "Dataset must have a column named 'english'"
# For scoring we need reference Hindi translations
assert 'hindi' in df.columns, "Dataset must have a column named 'hindi' with reference translations"

# Take the first N_SENTENCES (or fewer if the dataset is smaller)
df = df.head(N_SENTENCES).reset_index(drop=True)
english_sentences = df['english'].tolist()
reference_hindi = df['hindi'].tolist()

print(f"Loaded {len(english_sentences)} sentences.")


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)


def translate_batch(sentences, batch_size=8):
    """
    Translate a list of English sentences into Hindi.
    Returns a list of translated strings.
    """
    translations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        # Prepare the input with forced target language token
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        # Generate translations
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],
            max_length=128,
            num_beams=5,
            early_stopping=True
        )
        batch_translations = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
        translations.extend(batch_translations)
        print(f"Translated {min(i+batch_size, len(sentences))}/{len(sentences)}")
    return translations

# Run translation
print("Translating...")
model_translations = translate_batch(english_sentences, batch_size=8)


output_df = pd.DataFrame({
    'Original English sentence': english_sentences,
    'Model-generated Hindi translation': model_translations
})
output_df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Translations saved to {OUTPUT_EXCEL}")


# Reference is already tokenized Hindi text; sacrebleu handles tokenization internally.
# We need to present the references as a list of lists (one per sentence).
refs = [[ref] for ref in reference_hindi]  # sacrebleu expects list of reference lists

bleu = corpus_bleu(model_translations, refs)
chrf = corpus_chrf(model_translations, refs)
ter  = corpus_ter(model_translations, refs)

# Save scores to a text file
with open(OUTPUT_SCORES, "w", encoding="utf-8") as f:
    f.write(f"Number of sentences: {len(english_sentences)}\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write(f"BLEU score: {bleu.score:.2f}\n")
    f.write(f"chrF score: {chrf.score:.2f}\n")
    f.write(f"TER score:  {ter.score:.2f}\n")
    f.write("\nDetailed outputs:\n")
    f.write(f"BLEU: {bleu}\n")
    f.write(f"chrF: {chrf}\n")
    f.write(f"TER:  {ter}\n")

print(f"Scores saved to {OUTPUT_SCORES}")


print(f"\n--- Evaluation Scores ---")
print(f"BLEU: {bleu.score:.2f}")
print(f"chrF: {chrf.score:.2f}")
print(f"TER:  {ter.score:.2f}")

ModuleNotFoundError: No module named 'sacrebleu'